In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
import requests
import seaborn as sns

In [ ]:
# Constants / Configs
POSITION_MASTER = {1: 'GKP', 2: 'DEF', 3: 'MID', 4: 'FWD'}
PLAYERS_NUMERIC_COLUMNS = ['form', 'points_per_game', 'ep_next', 'influence', 'creativity', 'threat', 'ict_index', 
                           'value_form', 'value_season', 'selected_by_percent', 'expected_goals', 'expected_assists', 
                           'expected_goal_involvements', 'expected_goals_conceded', 'clean_sheets_per_90', 'saves_per_90', 'ppg_last']
PLAYERS_NORMALIZATION_COLUMNS = ['form', 'points_per_game', 'ep_next', 'fdr', 'ict_index', 'chance_of_playing_next_round', 
                                 'clean_sheets_per_90', 'saves_per_90', 'threat', 'ppg_last']

GKP_WEIGHTS = {
    'ep_next': 0.15,
    'fdr': 0.15,
    'chance_of_playing_next_round': 0.10,
    'clean_sheets_per_90': 0.275,
    'saves_per_90': 0.125,
}

DEF_WEIGHTS = {
    'ep_next': 0.20,
    'fdr': 0.15,
    'chance_of_playing_next_round': 0.05,
    'ict_index': 0.25,
    'clean_sheets_per_90': 0.15,
}

MID_WEIGHTS = {
    'ep_next': 0.20,
    'fdr': 0.18,
    'chance_of_playing_next_round': 0.10,
    'ict_index': 0.32,
}

FWD_WEIGHTS = {
    'ep_next': 0.225,
    'fdr': 0.1,
    'chance_of_playing_next_round': 0.10,
    'threat': 0.375,  # pick threat over expected_goals — forward-looking
}
PRESEASON_DATA_SHEET = 'GW0'

In [ ]:
# Get Next Gameweek
bootstrap_response = requests.get('https://fantasy.premierleague.com/api/bootstrap-static/')
bootstrap_response.raise_for_status()
bootstrap_data = bootstrap_response.json()
events = bootstrap_data["events"]

next_gw_title: str | None = None
next_gw_id: int = -1
curr_gw_id: int = 0

for event in events:
  if event["is_next"]:
    next_gw_id = event["id"]
    curr_gw_id = next_gw_id - 1
    next_gw_title = event["name"]
    break

if next_gw_id == -1:
  raise RuntimeError('No upcoming gameweek found.')

# The upcoming GW we are predicting for
# Sheet was fetched before this GW was played
GAMEWEEK = f'GW{curr_gw_id}'

In [ ]:
def get_season_weights(gws_played: int) -> tuple[float, float, float]:
  """Returns a tuple indicating the weight distribution of last season's points per game, and current season's points per game and 
  and form.
    
    (last_season_ppg, current_season_ppg, current_season_form)
    """
  if gws_played < 1:
    return 1, 0, 0
  elif gws_played < 4:
    return 0.7, 0.2, 0.1
  elif gws_played < 7:
    return 0.4, 0.3, 0.3
  elif gws_played < 11:
    return 0.1, 0.4, 0.5
  else:
    return 0, 0.45, 0.55

In [ ]:
# Load Data.
players_df = pd.read_excel('./data/players_master.xlsx', sheet_name=GAMEWEEK)
players_df_last_season = pd.read_excel('./data/players_master.xlsx', sheet_name='GW0')
teams_df = pd.read_excel('./data/teams_master.xlsx', sheet_name=GAMEWEEK)
fixtures_df = pd.read_excel('./data/fixtures_master.xlsx', sheet_name=GAMEWEEK)

In [ ]:
# Computations
players_df_last_season = players_df_last_season[['points_per_game', 'code']].rename(columns={'points_per_game': 'ppg_last'})
players_df = players_df.merge(players_df_last_season, on='code', how='left')
players_df['ppg_last'] = players_df['ppg_last'].fillna(0)
players_df['full_name'] = players_df['first_name'] + ' ' + players_df['second_name']
players_df['position'] = players_df['element_type'].map(POSITION_MASTER)
players_df['team_name'] = players_df['team'].map(dict(zip(teams_df['id'], teams_df['name'])))

# Using `errors='coerce'` to enter NaN for invalid values without raising exceptions.
for col in PLAYERS_NUMERIC_COLUMNS:
  players_df[col] = pd.to_numeric(players_df[col], errors='coerce')

# FPL passes values into 'chance_of_playing_next_round' only during issues example 50, 75.
# So missing value means player is fit to play. So replacing NA with 100 for calculations.
players_df['chance_of_playing_next_round'] = players_df['chance_of_playing_next_round'].fillna(100)
players_df['ep_next'] = players_df['ep_next'].fillna(0)
players_df = players_df[players_df['status'].isin(['a', 'i', 'd'])]

# Filtering out players with less than 900 minutes played.
players_df = players_df[players_df['minutes'] > 900]

# FPL sends cost (euro) multiplied for 10 by default.
players_df['cost'] = players_df['now_cost'] / 10
players_df['points_per_euro'] = players_df['total_points'] / players_df['cost']

home = fixtures_df[['team_h', 'team_h_difficulty']].rename(columns={'team_h': 'team_id', 'team_h_difficulty': 'fdr'})
away = fixtures_df[['team_a', 'team_a_difficulty']].rename(columns={'team_a': 'team_id', 'team_a_difficulty': 'fdr'})
fdr_df = pd.concat([home, away])
players_df['fdr'] = players_df['team'].map(dict(zip(fdr_df['team_id'], fdr_df['fdr'])))


# Creating a fresh contiguous copy of the DataFrame in memory as pandas internally fragments memory.
players_df = players_df.copy()

for col in PLAYERS_NORMALIZATION_COLUMNS:
  # FDR (Difficulty) Low = Easy (Good), High = Difficult (Bad)
  # Saver Per 90: Bad team's GK would have to go through more shots, inturn save more, but also creates possibility of conceiding more.
  # Inverting normalization for these as we want HIGH = Good, LOW = Bad.
  col_min = players_df[col].min()
  col_max = players_df[col].max()
  # Adding guard to avoid 0 in denominator
  denominator = col_max - col_min if col_max != col_min else 1
  players_df[f'{col}_norm'] = (players_df[col] - col_min) / denominator
  if col in ['fdr', 'saves_per_90' ]:
    players_df[f'{col}_norm'] = 1 - ((players_df[col] - col_min) / denominator)
  else:
    players_df[f'{col}_norm'] = (players_df[col] - col_min) / denominator

In [ ]:
# Scoring 

[ppg_last_w, ppg_curr_w, form_curr_w] = get_season_weights(curr_gw_id)

FORM_AND_PPG_WEIGHTS = {
  'ppg_last': 0.20 * ppg_last_w,
  'points_per_game': 0.20 * ppg_curr_w,
  'form': 0.20 * form_curr_w
}
gpk_weights = {**GKP_WEIGHTS, **FORM_AND_PPG_WEIGHTS}
def_weights = {**DEF_WEIGHTS, **FORM_AND_PPG_WEIGHTS}
mid_weights = {**MID_WEIGHTS, **FORM_AND_PPG_WEIGHTS}
fwd_weights = {**FWD_WEIGHTS, **FORM_AND_PPG_WEIGHTS}
 
mask = players_df['position'] == 'GKP'
players_df.loc[mask, 'next_gw_score'] = sum([players_df.loc[mask, f'{col}_norm'] * value for col, value in gpk_weights.items()])
mask = players_df['position'] == 'DEF'
players_df.loc[mask, 'next_gw_score'] = sum([players_df.loc[mask, f'{col}_norm'] * value for col, value in def_weights.items()])
mask = players_df['position'] == 'MID'
players_df.loc[mask, 'next_gw_score'] = sum([players_df.loc[mask, f'{col}_norm'] * value for col, value in mid_weights.items()])
mask = players_df['position'] == 'FWD'
players_df.loc[mask, 'next_gw_score'] = sum([players_df.loc[mask, f'{col}_norm'] * value for col, value in fwd_weights.items()])

# Players who actually play. (Score points)
active_players = players_df[players_df['total_points'] > 0]

In [ ]:
# Visualizations
plt.style.use('dark_background')

fig, ax = plt.subplots()
top_10_by_value = players_df.sort_values(by='points_per_euro', ascending=False).head(10).sort_values(by='points_per_euro', ascending=True)
ax.barh(top_10_by_value['full_name'], top_10_by_value['points_per_euro'])
ax.set_title('Top players: Points per Euro')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()
sns.kdeplot(data=active_players, x='ict_index', y='total_points', fill=True, ax=ax)
ax.set_title('ICT Index vs Total Points — Density')
plt.tight_layout()
plt.show()

In [ ]:
rec_cols = ['full_name', 'team_name', 'position', 'fdr', 'cost', 'total_points', 'points_per_euro', 'next_gw_score']
recommendations = players_df.sort_values(by=['position', 'next_gw_score'], ascending=False).groupby('position').head(10)[rec_cols]

for _, pos in POSITION_MASTER.items():
  temp_df = recommendations[recommendations['position'] == pos]
  display(
    temp_df.sort_values(by='next_gw_score', ascending=False)
    .style
    .set_caption(pos)
    .background_gradient(subset=['next_gw_score'], cmap='Greens')
    .format({'next_gw_score': '{:.2f}', 'cost': '£{:.1f}'})
  )